# Notebook 04 - Ian Research Memo

Research question: Can 5-minute bar data support an honest directional classifier that beats simple dummy baselines under chronological validation?

Scope: `validation_only` until explicitly changed. This notebook is the active research memo entry for the rebuilt project. It should stay readable to Ian and future reviewers without opening archived runner code, PM gates, or old helper-library plans.

Changed from default: none yet. This skeleton records the default `baseline_v1` setup and leaves every heavy cell disabled.

Do not claim: profitability, robustness, paper-ready evidence, model superiority, or holdout/test performance from this notebook skeleton.

## Setup And Run Guards

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd

from intraday_research import baseline_v1 as baseline_helpers

RUN_DATA_LOAD = False
RUN_FEATURE_BUILD = False
RUN_MODEL_VALIDATION = False
RUN_TRAINING = False

RESULT_SCOPE = "validation_only"
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd().resolve()
DATA_ROOT = PROJECT_ROOT / "data"

run_flags = {
    "run_data_load": RUN_DATA_LOAD,
    "run_feature_build": RUN_FEATURE_BUILD,
    "run_model_validation": RUN_MODEL_VALIDATION,
    "run_training": RUN_TRAINING,
    "result_scope": RESULT_SCOPE,
}

run_flags

## Baseline Configuration

In [ ]:
TICKERS = ["CSCO", "JPM", "KO", "MSFT", "WMT"]
FEATURE_SET_ID = "baseline_v1"
FEATURE_COLUMNS = [
    "log_return",
    "close_to_open_return",
    "high_low_range",
    "rolling_volatility_20",
    "normalized_volume_20",
    "rsi_14",
    "bollinger_pctb",
    "normalized_macd_hist",
    "time_of_day_sin",
    "time_of_day_cos",
]

LABEL_POLICY = "no_trade_band"
THRESHOLD_SOURCE = "fixed_pre_registered_5bps"
THRESHOLD_BPS = 5.0
LABEL_HORIZON_K = 12
WINDOW_SIZE = 12
DECISION_TIME_POLICY = "post_bar_close_completed_bar"
SCALER_ID = "standard_pooled_train_only_v1"
SCALER_FIT_SCOPE = "pooled_train_after_per_ticker_chronological_split"

CALENDAR_SPLITS = {
    "train": ("1998-01-02", "2013-09-16"),
    "validation": ("2013-09-16", "2017-01-25"),
    "closed_holdout_boundary_only": ("2017-01-25", "2020-06-06"),
}

baseline_metadata = {
    "feature_set_id": FEATURE_SET_ID,
    "label_policy": LABEL_POLICY,
    "threshold_source": THRESHOLD_SOURCE,
    "threshold_bps": THRESHOLD_BPS,
    "label_horizon_k": LABEL_HORIZON_K,
    "window_size": WINDOW_SIZE,
    "decision_time_policy": DECISION_TIME_POLICY,
    "scaler_id": SCALER_ID,
    "scaler_fit_scope": SCALER_FIT_SCOPE,
    "scope": RESULT_SCOPE,
}

baseline_metadata

## Data Contract And Paths

Expected raw input: one 5-minute OHLCV CSV per ticker under `data/`.

Expected columns:

| Field | Purpose |
|---|---|
| timestamp | chronological ordering and trading-day grouping |
| open | completed-bar open price |
| high | completed-bar high price |
| low | completed-bar low price |
| close | completed-bar close price |
| volume | completed-bar volume |

Read-only coverage audit on 2026-06-02:

| Ticker | File | Rows | Start timestamp | End timestamp | Missing total | Duplicate timestamps |
|---|---|---:|---|---|---:|---:|
| CSCO | `data/CSCO.csv` | 444305 | 1998-01-02 09:30:00 | 2020-06-08 16:00:00 | 0 | 0 |
| JPM | `data/JPM.csv` | 443589 | 1998-01-02 09:30:00 | 2020-06-05 16:00:00 | 0 | 0 |
| KO | `data/KO.csv` | 443273 | 1998-01-02 09:30:00 | 2020-06-08 16:00:00 | 0 | 0 |
| MSFT | `data/MSFT.csv` | 444322 | 1998-01-02 09:30:00 | 2020-06-05 16:00:00 | 0 | 0 |
| WMT | `data/WMT.csv` | 443278 | 1998-01-02 09:30:00 | 2020-06-08 16:00:00 | 0 | 0 |

This audit reads only raw file shape and timestamp coverage. It does not train,
score models, inspect old checkpoints, or read holdout/test metrics.

## Load And Coverage Diagnostics

In [ ]:
def find_timestamp_column(columns):
    for candidate in ("timestamp", "datetime", "date", "time"):
        for column in columns:
            if column.lower() == candidate:
                return column
    raise ValueError(f"No timestamp-like column found in columns: {list(columns)}")


def load_ticker_csv(ticker, data_root=DATA_ROOT):
    path = data_root / f"{ticker}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing raw ticker file: {path}")
    frame = pd.read_csv(path)
    timestamp_col = find_timestamp_column(frame.columns)
    frame = frame.rename(columns={timestamp_col: "timestamp"}).copy()
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], errors="raise")
    frame["ticker"] = ticker
    return frame.sort_values("timestamp").reset_index(drop=True)


def audit_ticker_frame(ticker, frame):
    missing = int(frame[expected_columns].isna().sum().sum())
    return {
        "ticker": ticker,
        "path": str(expected_data_files[ticker]),
        "exists": expected_data_files[ticker].exists(),
        "n_rows": int(len(frame)),
        "start_ts": str(frame["timestamp"].min()),
        "end_ts": str(frame["timestamp"].max()),
        "missing_total": missing,
        "duplicate_timestamp_count": int(frame["timestamp"].duplicated().sum()),
    }


def audit_all_tickers(tickers=TICKERS):
    frames = {ticker: load_ticker_csv(ticker) for ticker in tickers}
    coverage = pd.DataFrame(
        [audit_ticker_frame(ticker, frame) for ticker, frame in frames.items()]
    )
    return frames, coverage


expected_data_files = {ticker: DATA_ROOT / f"{ticker}.csv" for ticker in TICKERS}
expected_columns = ["timestamp", "open", "high", "low", "close", "volume"]
coverage_columns = [
    "ticker",
    "path",
    "exists",
    "n_rows",
    "start_ts",
    "end_ts",
    "missing_total",
    "duplicate_timestamp_count",
    "audit_source",
    "current_verification",
    "current_file_exists",
]

observed_data_coverage = pd.DataFrame([
    {"ticker": "CSCO", "path": "data/CSCO.csv", "exists": True, "n_rows": 444305, "start_ts": "1998-01-02 09:30:00", "end_ts": "2020-06-08 16:00:00", "missing_total": 0, "duplicate_timestamp_count": 0},
    {"ticker": "JPM", "path": "data/JPM.csv", "exists": True, "n_rows": 443589, "start_ts": "1998-01-02 09:30:00", "end_ts": "2020-06-05 16:00:00", "missing_total": 0, "duplicate_timestamp_count": 0},
    {"ticker": "KO", "path": "data/KO.csv", "exists": True, "n_rows": 443273, "start_ts": "1998-01-02 09:30:00", "end_ts": "2020-06-08 16:00:00", "missing_total": 0, "duplicate_timestamp_count": 0},
    {"ticker": "MSFT", "path": "data/MSFT.csv", "exists": True, "n_rows": 444322, "start_ts": "1998-01-02 09:30:00", "end_ts": "2020-06-05 16:00:00", "missing_total": 0, "duplicate_timestamp_count": 0},
    {"ticker": "WMT", "path": "data/WMT.csv", "exists": True, "n_rows": 443278, "start_ts": "1998-01-02 09:30:00", "end_ts": "2020-06-08 16:00:00", "missing_total": 0, "duplicate_timestamp_count": 0},
])

raw_frames = {}
if RUN_DATA_LOAD:
    raw_frames, observed_data_coverage = audit_all_tickers()
    observed_data_coverage["audit_source"] = "current_run"
    observed_data_coverage["current_verification"] = True
    observed_data_coverage["current_file_exists"] = observed_data_coverage["exists"]
else:
    observed_data_coverage["audit_source"] = "prior_read_only_shape_audit"
    observed_data_coverage["current_verification"] = False
    observed_data_coverage["current_file_exists"] = observed_data_coverage["ticker"].map(
        lambda ticker: expected_data_files[ticker].exists()
    )
    print("Data loading disabled. Static row counts and timestamps are from a prior read-only shape audit; current_file_exists is the only live check in this guarded view.")

observed_data_coverage

## Causal Feature Construction

In [ ]:
feature_definition_table = pd.DataFrame([
    {"feature": "log_return", "basis": "log close at t minus log close at t-1", "availability": "after completed bar close"},
    {"feature": "close_to_open_return", "basis": "close[t] / open[t] - 1", "availability": "after completed bar close"},
    {"feature": "high_low_range", "basis": "log(high[t] / low[t])", "availability": "after completed bar close"},
    {"feature": "rolling_volatility_20", "basis": "prior 20-bar rolling std of intraday log_return; first valid at 0-based bar 21", "availability": "after completed bar close"},
    {"feature": "normalized_volume_20", "basis": "log1p(volume[t]) minus prior 20-bar same-day mean; first valid at 0-based bar 20", "availability": "after completed bar close"},
    {"feature": "rsi_14", "basis": "14-bar Wilder-style RSI from completed closes", "availability": "after completed bar close"},
    {"feature": "bollinger_pctb", "basis": "current close relative to trailing Bollinger band; zero-width bands are invalid", "availability": "after completed bar close"},
    {"feature": "normalized_macd_hist", "basis": "MACD histogram normalized by EMA-26", "availability": "after completed bar close"},
    {"feature": "time_of_day_sin", "basis": "sine transform of minutes since regular-market open over a 390-minute session", "availability": "known before bar"},
    {"feature": "time_of_day_cos", "basis": "cosine transform of minutes since regular-market open over a 390-minute session", "availability": "known before bar"},
])

features_by_ticker = {}
if RUN_FEATURE_BUILD:
    source_frames = raw_frames or audit_all_tickers()[0]
    features_by_ticker = {
        ticker: baseline_helpers.add_baseline_v1_features(frame)
        for ticker, frame in source_frames.items()
    }
else:
    print("Feature construction disabled. Flip the approved feature-build guard only for an approved validation-only task.")

feature_definition_table

## Label Construction And Invalid Markers

In [ ]:
label_columns = [
    "future_cumulative_return",
    "label",
    "invalid_cross_day",
    "invalid_missing_future",
]

label_policy_table = pd.DataFrame([{
    "label_policy": LABEL_POLICY,
    "threshold_bps": THRESHOLD_BPS,
    "threshold_source": THRESHOLD_SOURCE,
    "horizon_k": LABEL_HORIZON_K,
    "invalid_marker_policy": "invalid labels remain markers until split/day/window validity is enforced",
    "helper_source": "intraday_research.baseline_v1.make_no_trade_band_labels",
}])

labeled_by_ticker = {}
if RUN_FEATURE_BUILD and features_by_ticker:
    labeled_by_ticker = {
        ticker: baseline_helpers.make_no_trade_band_labels(
            frame,
            horizon_k=LABEL_HORIZON_K,
            threshold_bps=THRESHOLD_BPS,
        )
        for ticker, frame in features_by_ticker.items()
    }

label_policy_table

## Chronological Split And Split-Boundary Invalidation

In [ ]:
split_boundary_checks = pd.DataFrame(columns=[
    "ticker",
    "split",
    "n_rows",
    "valid_labels",
    "invalid_cross_split",
    "invalid_cross_day",
    "invalid_missing_future",
])

split_rules = {
    "chronological_per_ticker": True,
    "invalidate_horizon_crossing_split": True,
    "do_not_globally_drop_invalid_labels_before_boundary_checks": True,
    "closed_holdout_boundary_only": "available for boundary invalidation only",
    "closed_holdout_never_scored_transformed_or_used_for_selection": True,
    "helper_source": "intraday_research.baseline_v1.add_split_and_invalidate_boundaries",
}

split_by_ticker = {}
if RUN_FEATURE_BUILD and labeled_by_ticker:
    split_by_ticker = {
        ticker: baseline_helpers.add_split_and_invalidate_boundaries(
            frame,
            splits=CALENDAR_SPLITS,
            horizon_k=LABEL_HORIZON_K,
        )
        for ticker, frame in labeled_by_ticker.items()
    }

split_rules

## Train-Only Preprocessing

In [ ]:
preprocessing_rules = {
    "scaler_id": SCALER_ID,
    "fit_scope": SCALER_FIT_SCOPE,
    "fit_rows": "train rows only, pooled after per-ticker chronological split",
    "transform_rows": "train and validation rows only in validation-only work",
    "closed_holdout_transform_policy": "forbidden; boundary invalidation only",
    "helper_source": "intraday_research.baseline_v1.fit_train_only_scaler / transform_train_and_validation",
    "forbidden": [
        "fit on full sample",
        "fit before chronological split",
        "fit on validation or closed holdout/test rows",
        "global full-sample normalization",
    ],
}

scaler = None
scaled_by_ticker = {}
if RUN_FEATURE_BUILD and split_by_ticker:
    scaler = baseline_helpers.fit_train_only_scaler(
        split_by_ticker,
        feature_columns=FEATURE_COLUMNS,
    )
    scaled_by_ticker = baseline_helpers.transform_train_and_validation(
        split_by_ticker,
        scaler,
        feature_columns=FEATURE_COLUMNS,
    )

preprocessing_rules

## Per-Ticker Per-Split Window Construction

In [ ]:
window_schema = {
    "window_size": WINDOW_SIZE,
    "label_horizon_k": LABEL_HORIZON_K,
    "unit": "single ticker and single split segment",
    "helper_source": "intraday_research.baseline_v1.build_windows_by_ticker_and_split",
    "forbidden": [
        "window spanning tickers",
        "window spanning train/validation or validation/closed-holdout boundary",
        "window spanning trading days",
        "raw feature fallback for model windows",
    ],
}

windows_by_ticker = {}
if RUN_FEATURE_BUILD and scaled_by_ticker:
    windows_by_ticker = baseline_helpers.build_windows_by_ticker_and_split(
        scaled_by_ticker,
        feature_columns=FEATURE_COLUMNS,
        window_size=WINDOW_SIZE,
    )

window_schema

## Target And Class Balance Diagnostics

In [ ]:
def summarize_window_class_balance(windows):
    rows = []
    for ticker, split_map in windows.items():
        for split_name, bundle in split_map.items():
            labels = bundle["y"]
            rows.append(
                {
                    "ticker": ticker,
                    "split": split_name,
                    "n_windows": int(len(labels)),
                    "class_0": int((labels == 0).sum()),
                    "class_1": int((labels == 1).sum()),
                    "invalid_labels": 0,
                    "skipped_windows": None,
                    "scope": RESULT_SCOPE,
                }
            )
    return pd.DataFrame(rows)


class_balance_schema = pd.DataFrame(columns=[
    "ticker",
    "split",
    "n_windows",
    "class_0",
    "class_1",
    "invalid_labels",
    "skipped_windows",
    "scope",
])

class_balance_table = (
    summarize_window_class_balance(windows_by_ticker)
    if windows_by_ticker
    else class_balance_schema.copy()
)

class_balance_table

## Stratified Dummy Baselines

In [ ]:
def pooled_train_validation_labels(windows):
    train_labels, validation_labels = [], []
    for split_map in windows.values():
        train_labels.append(split_map["train"]["y"])
        validation_labels.append(split_map["validation"]["y"])
    return np.concatenate(train_labels), np.concatenate(validation_labels)


dummy_baseline_schema = pd.DataFrame(columns=[
    "model",
    "ticker_or_pooled",
    "dummy_strategy",
    "seed",
    "macro_f1",
    "balanced_accuracy",
    "accuracy",
    "n",
    "scope",
])

dummy_baseline_table = dummy_baseline_schema.copy()
if RUN_MODEL_VALIDATION and windows_by_ticker:
    y_train_pooled, y_validation_pooled = pooled_train_validation_labels(windows_by_ticker)
    dummy_baseline_table = baseline_helpers.evaluate_stratified_dummy(
        y_train_pooled,
        y_validation_pooled,
    )
    dummy_baseline_table.insert(0, "model", "stratified_dummy")
    dummy_baseline_table.insert(1, "ticker_or_pooled", "pooled")
    dummy_baseline_table.insert(2, "dummy_strategy", "stratified")
    dummy_baseline_table = dummy_baseline_table.rename(columns={"validation_n": "n"})
    dummy_baseline_table["scope"] = RESULT_SCOPE
else:
    print("Dummy baseline evaluation disabled until approved validation target arrays exist.")

dummy_baseline_table

## Model Availability And Validation Plan

LightGBM status: archive contains validation-only reference logic, and `requirements.txt` includes LightGBM. Active use still requires a migration audit and an approved validation-only task.

MS-DLinear+TCN status: archive contains model and test references. Active use still requires shape/spec audit, adapter review, and an approved validation-only task.

Do not run archived CLI paths as the active workflow. Do not report old checkpoint results as current notebook evidence.

## Validation-Only Model Cells

In [ ]:
model_result_schema = pd.DataFrame(columns=[
    "model",
    "ticker_or_pooled",
    "macro_f1",
    "balanced_accuracy",
    "dummy_macro_f1",
    "delta_macro_f1_vs_dummy",
    "n",
    "scope",
])

if not RUN_MODEL_VALIDATION:
    print("Model validation is disabled. Flip the approved model-validation guard only in an approved validation-only task.")
else:
    raise NotImplementedError("Active model adapter migration is required before validation.")

model_result_schema

## Comparison Table

In [ ]:
comparison_columns = [
    "model",
    "ticker_or_pooled",
    "macro_f1",
    "balanced_accuracy",
    "dummy_macro_f1",
    "delta_macro_f1_vs_dummy",
    "n",
    "scope",
]

comparison_table = pd.DataFrame(columns=comparison_columns)
comparison_table

## Validation Plots

In [ ]:
plot_plan = pd.DataFrame([
    {"plot": "delta_macro_f1_vs_dummy_by_ticker_model", "requires": "comparison_table", "scope": RESULT_SCOPE},
    {"plot": "class_balance_by_ticker_split", "requires": "class_balance_schema", "scope": RESULT_SCOPE},
])

plot_plan

## Ian-Facing Interpretation

Evidence: summarize what was actually tested, including split policy, feature set, label policy, scaler fit scope, dummy baselines, and validation-only metrics.

Caveats: state what failed, what remains diagnostic, where class balance or coverage is weak, and what this notebook does not support.

Next action: propose one concrete next research step without using holdout/test, old checkpoints, or archived runner outputs as current evidence.